## 1. import package

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt


from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.ensemble import RandomForestClassifier  #ensemble method
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


from IPython.display import display
import cv2

from skimage.feature import graycomatrix, graycoprops


import torch
import torchvision.transforms as T
from torchvision.models import efficientnet_b2, efficientnet_b4, EfficientNet_B2_Weights, EfficientNet_B4_Weights



import warnings
warnings.filterwarnings("ignore")
# Portfolio reproducibility seeds (original metrics are not regenerated).
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


## 2. load data

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
Task1_DIR = PROJECT_ROOT / 'task1_data'
Task2_DIR = PROJECT_ROOT / 'task2_data'

### 2.1 Task1 Data 

In [ ]:
# metadata
task1_train_metadata = pd.read_csv(Task1_DIR / 'train_metadata.csv')
task1_test_metadata = pd.read_csv(Task1_DIR / 'test_metadata.csv')

#features
task1_color = pd.read_csv(Task1_DIR / 'color_histogram.csv')
task1_hog = pd.read_csv(Task1_DIR / 'hog_pca.csv')
task1_additional = pd.read_csv(Task1_DIR/'additional_features.csv')

### 2.2 Task2 Data 

In [ ]:
# metadata
task2_train_metadata = pd.read_csv(Task2_DIR / 'train_metadata.csv')
task2_test_metadata = pd.read_csv(Task2_DIR / 'test_metadata.csv')
task2_class_mapping = pd.read_csv(Task2_DIR / 'class_mapping.csv')

#features
task2_color = pd.read_csv(Task2_DIR / 'color_histogram.csv')
task2_hog = pd.read_csv(Task2_DIR / 'hog_pca.csv')
task2_additional = pd.read_csv(Task2_DIR / 'additional_features.csv')

## 3. data preprocessing

### 3.1 Combine Metadata and Feature Sets

In [ ]:
# task 1
task1_train_full = (
    task1_train_metadata
    .merge(task1_color, on='image_id')
    .merge(task1_hog, on='image_id')
    .merge(task1_additional, on='image_id')
)

task1_test_full = (
    task1_test_metadata
    .merge(task1_color, on='image_id')
    .merge(task1_hog, on='image_id')
    .merge(task1_additional, on='image_id')
)


# Task 2
task2_train_full = (
    task2_train_metadata
    .merge(task2_color, on='image_id')
    .merge(task2_hog, on='image_id')
    .merge(task2_additional, on='image_id')
)

task2_test_full = (
    task2_test_metadata
    .merge(task2_color, on='image_id')
    .merge(task2_hog, on='image_id')
    .merge(task2_additional, on='image_id')
)

### 3.2 Train / Validation Split

In [ ]:
# Task 1: separate features and label
task1_X = task1_train_full.drop(
    columns=['image_id', 'image_path', 'class_id', 'class_name']
)

task1_y = task1_train_full['class_name']


# Task 2: separate features and label
task2_X = task2_train_full.drop(
    columns=['image_id', 'image_path', 'class_id', 'class_name']
)

task2_y = task2_train_full['class_name']


# Task 1: split into training and validation sets
task1_X_train, task1_X_valid, task1_y_train, task1_y_valid = train_test_split(
    task1_X,
    task1_y,
    test_size=0.2,
    random_state=42,
    stratify=task1_y
)


# Task 2: split into training and validation sets
task2_X_train, task2_X_valid, task2_y_train, task2_y_valid = train_test_split(
    task2_X,
    task2_y,
    test_size=0.25,
    random_state=42,
    stratify=task2_y
)

### Feature Standardisation

In [ ]:
# Task 1
task1_scaler = StandardScaler()

task1_X_train_scaled = task1_scaler.fit_transform(
    task1_X_train
)

task1_X_valid_scaled = task1_scaler.transform(
    task1_X_valid
)


# Task 2
task2_scaler = StandardScaler()

task2_X_train_scaled = task2_scaler.fit_transform(
    task2_X_train
)

task2_X_valid_scaled = task2_scaler.transform(
    task2_X_valid
)

## 4. Baseline Experiments

### Model Evaluation Utilities

In [ ]:
def fit_and_evaluate_model(model, X_train, y_train, X_valid, y_valid, model_name):

    model_copy = clone(model)
    model_copy.fit(X_train, y_train)

    train_pred = model_copy.predict(X_train)
    valid_pred = model_copy.predict(X_valid)

    class_names = sorted(y_valid.unique())

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_valid,
        valid_pred,
        labels=class_names,
        average=None,
        zero_division=0
    )

    class_results = pd.DataFrame({
        'class_name': class_names,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

    worst_precision = class_results.loc[class_results['precision'].idxmin()]
    worst_recall = class_results.loc[class_results['recall'].idxmin()]
    worst_f1 = class_results.loc[class_results['f1'].idxmin()]

    result = pd.DataFrame([{
        'model_name': model_name,

        'train_accuracy': accuracy_score(y_train, train_pred),
        'valid_accuracy': accuracy_score(y_valid, valid_pred),

        'valid_precision': class_results['precision'].mean(),
        'valid_recall': class_results['recall'].mean(),
        'valid_f1': class_results['f1'].mean(),

        'lowest_precision': worst_precision['precision'],
        'lowest_precision_class': worst_precision['class_name'],

        'lowest_recall': worst_recall['recall'],
        'lowest_recall_class': worst_recall['class_name'],

        'lowest_f1': worst_f1['f1'],
        'lowest_f1_class': worst_f1['class_name'],

        'fitted_model': model_copy,
        'class_results': class_results
    }])

    return result

In [ ]:
def run_baseline_models(
    baseline_models,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_train_scaled,
    X_valid_scaled
):

    model_results = []

    for model_name, model in baseline_models.items():

        if model_name == 'random_forest':

            result = fit_and_evaluate_model(
                model,
                X_train,
                y_train,
                X_valid,
                y_valid,
                model_name
            )

        else:

            result = fit_and_evaluate_model(
                model,
                X_train_scaled,
                y_train,
                X_valid_scaled,
                y_valid,
                model_name
            )

        model_results.append(result)

    model_results = pd.concat(
        model_results,
        ignore_index=True
    ).sort_values(
        ['valid_f1', 'valid_accuracy'],
        ascending=False
    ).reset_index(drop=True)

    return model_results

In [ ]:
def display_model_results(model_results):

    display_columns = [
        'model_name',
        'train_accuracy',
        'valid_accuracy',
        'valid_precision',
        'valid_recall',
        'valid_f1',
        'lowest_f1',
        'lowest_f1_class'
    ]
    result = model_results[display_columns]
    display(result)
    print(result.round(4).to_latex(index=False))

In [ ]:
def display_worst_class_analysis(model_results):

    columns = [
        'model_name',
        'lowest_f1',
        'lowest_f1_class',
        'lowest_recall',
        'lowest_recall_class',
        'lowest_precision',
        'lowest_precision_class'
    ]
    result = model_results[columns].sort_values('lowest_f1').reset_index(drop=True)
    display(result)
    print(result.round(4).to_latex(index=False))

In [ ]:
def display_overfitting_analysis(model_results):

    model_results = model_results.copy()
    model_results['overfit_gap'] = (
        model_results['train_accuracy']
        - model_results['valid_accuracy']
    ).round(3)

    result = model_results[
            [
                'model_name',
                'train_accuracy',
                'valid_accuracy',
                'overfit_gap'
            ]
        ].sort_values(
            'overfit_gap',
            ascending=False
        ).reset_index(drop=True)
    display(result)
    print(result.round(4).to_latex(index=False))

### 4.1 Selected Baseline Models

In [ ]:
task1_models = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=42),

    'linear_svm': LinearSVC(random_state=42, max_iter=20000, dual='auto'),

    'rbf_svm': SVC(kernel='rbf', C=1.0, gamma='scale'),

    'random_forest': RandomForestClassifier(n_estimators=200, random_state=42)
}


task2_models = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=42),

    'linear_svm': LinearSVC(random_state=42, max_iter=20000, dual='auto'),

    'rbf_svm': SVC(kernel='rbf', C=1.0, gamma='scale', probability=True),

}

### 4.2 Task 1 Baseline Results

In [ ]:
task1_model_results = run_baseline_models(task1_models,task1_X_train,task1_y_train,task1_X_valid,task1_y_valid,task1_X_train_scaled,task1_X_valid_scaled)

In [ ]:
display_model_results(task1_model_results)

In [ ]:
display_worst_class_analysis(task1_model_results)

In [ ]:
display_overfitting_analysis(task1_model_results)

### 4.3 Task 2 Baseline Results

In [ ]:
task2_model_results = run_baseline_models(task2_models,task2_X_train,task2_y_train,task2_X_valid,task2_y_valid,task2_X_train_scaled,task2_X_valid_scaled)

In [ ]:
display_model_results(task2_model_results)

In [ ]:
display_worst_class_analysis(task2_model_results)

In [ ]:
display_overfitting_analysis(task2_model_results)

### 4.4 Task1 vs Task2 comparison

#### 1. performance

In [ ]:
# comparison of Task 1 and Task 2 model performance
def compare_tasks(
    task1_results,
    task2_results
):

    comparison_table = (
        task1_results[
            ['model_name', 'valid_f1']
        ].rename(
            columns={
                'valid_f1': 'task1_f1'
            }
        )
        .merge(
            task2_results[
                ['model_name', 'valid_f1']
            ].rename(
                columns={
                    'valid_f1': 'task2_f1'
                }
            ),
            on='model_name'
        )
    )

    comparison_table['performance_drop'] = (
        comparison_table['task1_f1']
        - comparison_table['task2_f1']
    ).round(3)

    return comparison_table.sort_values(
        'performance_drop',
        ascending=False
    ).reset_index(drop=True)

# create Task 1 vs Task 2 comparison table
task_comparison = compare_tasks(task1_model_results,task2_model_results)
display(task_comparison)
print((task_comparison).to_latex(index=False))

In [ ]:
# plot Task 1 vs Task 2 F1 comparison

plot_data = task_comparison[
    ['model_name', 'task1_f1', 'task2_f1']
].set_index('model_name')

ax = plot_data.plot(
    kind='bar',
    figsize=(8, 5)
)

ax.set_title('Task 1 vs Task 2 Baseline Model Performance')
ax.set_xlabel('Model')
ax.set_ylabel('Validation F1 Score')
ax.set_ylim(0, 1)
ax.legend(['Task 1', 'Task 2'])

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

#### 2. overfitting

In [ ]:
def compare_overfitting(
    task1_results,
    task2_results
):

    task1_gap = task1_results[
        ['model_name', 'train_accuracy', 'valid_accuracy']
    ].copy()

    task1_gap['task1_overfit_gap'] = (
        task1_gap['train_accuracy']
        - task1_gap['valid_accuracy']
    ).round(3)

    task2_gap = task2_results[
        ['model_name', 'train_accuracy', 'valid_accuracy']
    ].copy()

    task2_gap['task2_overfit_gap'] = (
        task2_gap['train_accuracy']
        - task2_gap['valid_accuracy']
    ).round(3)

    overfit_comparison = (
        task1_gap[
            ['model_name', 'task1_overfit_gap']
        ].merge(
            task2_gap[
                ['model_name', 'task2_overfit_gap']
            ],
            on='model_name'
        )
    )

    overfit_comparison['gap_increase'] = (
        overfit_comparison['task2_overfit_gap']
        - overfit_comparison['task1_overfit_gap']
    ).round(3)

    return overfit_comparison.sort_values(
        'gap_increase',
        ascending=False
    ).reset_index(drop=True)

overfit_comparison = compare_overfitting(
    task1_model_results,
    task2_model_results
)

display(overfit_comparison)

In [ ]:
plot_data = overfit_comparison[
    [
        'model_name',
        'task1_overfit_gap',
        'task2_overfit_gap'
    ]
].set_index('model_name')

ax = plot_data.plot(
    kind='bar',
    figsize=(8, 5)
)

ax.set_title('Task 1 vs Task 2 Overfitting Comparison')
ax.set_xlabel('Model')
ax.set_ylabel('Train-Validation Accuracy Gap')

ax.legend([
    'Task 1',
    'Task 2'
])

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. error analysis

### build validation table

In [ ]:
# build validation prediction table
def build_validation_predictions(
    model_row,
    full_train_table,
    X_valid,
    X_valid_scaled,
    y_valid
):

    # get validation image path and true class
    validation_table = full_train_table.loc[
        X_valid.index,
        [
            'image_path',
            'class_name'
        ]
    ].copy()

    # rename true label column
    validation_table = validation_table.rename(
        columns={
            'class_name': 'true_class'
        }
    )

    # use scaled data for linear models and SVM
    if model_row['model_name'] in [
        'logistic_regression',
        'linear_svm',
        'rbf_svm'
    ]:
        X_eval = X_valid_scaled

    # use unscaled data for random forest
    else:
        X_eval = X_valid

    # predict validation labels
    validation_table['predicted_class'] = (
        model_row['fitted_model'].predict(X_eval)
    )

    # check whether prediction is correct
    validation_table['is_correct'] = (
        validation_table['true_class']
        ==
        validation_table['predicted_class']
    )

    return validation_table

### pick best performance model for each task for demonstration purpose

In [ ]:
# build Task 1 validation prediction table

task1_best_model_row = task1_model_results.iloc[0]

task1_validation_predictions = build_validation_predictions(
    task1_best_model_row,
    task1_train_full,
    task1_X_valid,
    task1_X_valid_scaled,
    task1_y_valid
)


In [ ]:
# build Task 2 validation prediction table

task2_best_model_row = task2_model_results.iloc[0]

task2_validation_predictions = build_validation_predictions(
    task2_best_model_row,
    task2_train_full,
    task2_X_valid,
    task2_X_valid_scaled,
    task2_y_valid
)


### 5.1 Prediction Distribution Analysis

In [ ]:
# task1_validation_predictions['predicted_class'].value_counts()
# task2_validation_predictions['predicted_class'].value_counts()

# Task 2 prediction distribution table with percentage
# lower class are poorly predicted (class imbalance issue)

task2_prediction_distribution = (
    task2_validation_predictions['predicted_class']
    .value_counts()
    .reset_index()
)

task2_prediction_distribution.columns = [
    'predicted_class',
    'count'
]

task2_prediction_distribution['percentage'] = (
    task2_prediction_distribution['count']
    / task2_prediction_distribution['count'].sum()
    * 100
).round(2)

display(task2_prediction_distribution) 

### 5.2 confusion matrix

In [ ]:
def plot_confusion_matrix(true_labels, pred_labels, title, cmap='Blues', figsize=(12, 10), save_path = None):
    class_order = sorted(set(true_labels))
    short_labels = [c.replace('_', ' ') for c in class_order]

    cm = confusion_matrix(true_labels, pred_labels,
                          labels=class_order, normalize='true')

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(range(len(class_order)))
    ax.set_yticks(range(len(class_order)))
    ax.set_xticklabels(short_labels, rotation=40, ha='right', fontsize=10)
    ax.set_yticklabels(short_labels, fontsize=10)

    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i, j]:.2f}',
                    ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > thresh else 'black')

    ax.set_xlabel('Predicted Label', fontsize=12, labelpad=10)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(title, fontsize=13, pad=15)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")

    plt.show()

In [ ]:
# plot confusion matrices for Task 1 and Task 2
task1_class_order = sorted(
    task1_validation_predictions['true_class'].unique()
)

task2_class_order = sorted(
    task2_validation_predictions['true_class'].unique()
)


In [ ]:
# Task 1 confusion matrix
plot_confusion_matrix(
    task1_validation_predictions['true_class'],
    task1_validation_predictions['predicted_class'],
    title=f"Task 1 Confusion Matrix — {task1_best_model_row['model_name']}",
    cmap='Blues',
    save_path= "T1 Confusion Matrix"
)

In [ ]:
# Task 2 confusion matrix
plot_confusion_matrix(
    task2_validation_predictions['true_class'],
    task2_validation_predictions['predicted_class'],
    title=f"Task 2 Confusion Matrix — {task2_best_model_row['model_name']}",
    cmap='Oranges',
    figsize=(13, 11),
    save_path= "T2 Confusion Matrix"
)

### 5.3 Misclassification Analysis

#### misclassification Analysis untilities

In [ ]:
def get_confusion_pairs(validation_predictions):

    confusion_pairs = (
        validation_predictions.loc[
            ~validation_predictions['is_correct'],
            ['true_class', 'predicted_class']
        ]
        .value_counts()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .reset_index(drop=True)
    )

    return confusion_pairs

In [ ]:
def show_confused_vs_reference_examples(

    validation_predictions,
    confusion_pairs,
    image_folder,
    confusion_pair_index=0,
    n_examples=3

):

    # select confusion pair
    confusion_pair = confusion_pairs.iloc[
        confusion_pair_index
    ]

    true_class = confusion_pair[
        'true_class'
    ]

    predicted_class = confusion_pair[
        'predicted_class'
    ]


    # confused examples
    confused_examples = (
        validation_predictions.loc[
            (
                validation_predictions['true_class']
                == true_class
            )
            &
            (
                validation_predictions['predicted_class']
                == predicted_class
            ),
            [
                'image_path',
                'true_class',
                'predicted_class'
            ]
        ]
        .head(n_examples)
        .reset_index(drop=True)
    )


    # reference examples
    reference_examples = (
        validation_predictions.loc[
            (
                validation_predictions['true_class']
                == predicted_class
            )
            &
            (
                validation_predictions['predicted_class']
                == predicted_class
            ),
            [
                'image_path',
                'true_class',
                'predicted_class'
            ]
        ]
        .head(n_examples)
        .reset_index(drop=True)
    )


    # add image paths
    confused_examples['image_path'] = (
        image_folder + '/'
        + confused_examples['image_path']
    )

    reference_examples['image_path'] = (
        image_folder + '/'
        + reference_examples['image_path']
    )


    # display confusion pair
    display(
        confusion_pair.to_frame().T
    )


    # create figure
    fig, axes = plt.subplots(
        2,
        n_examples,
        figsize=(4 * n_examples, 8)
    )

    axes = axes.ravel()


    # plot confused examples
    for i, ax in enumerate(axes[:n_examples]):

        ax.axis('off')

        if i < len(confused_examples):

            row = confused_examples.loc[i]

            image = Image.open(
                row['image_path']
            )

            ax.imshow(image)

            ax.set_title(
                f"confused\ntrue: {row['true_class']}\npred: {row['predicted_class']}",
                fontsize=10
            )


    # plot reference examples
    for i, ax in enumerate(axes[n_examples:]):

        ax.axis('off')

        if i < len(reference_examples):

            row = reference_examples.loc[i]

            image = Image.open(
                row['image_path']
            )

            ax.imshow(image)

            ax.set_title(
                f"reference\ntrue: {row['true_class']}\npred: {row['predicted_class']}",
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

#### Task 1 confusion pairs && misclassified and reference examples

In [ ]:
# Task 1 confusion pairs table
task1_confusion_pairs = get_confusion_pairs(
    task1_validation_predictions
)

display(task1_confusion_pairs.head(5))

In [ ]:
show_confused_vs_reference_examples(
    task1_validation_predictions,
    task1_confusion_pairs,
    image_folder= str(Task1_DIR),
    confusion_pair_index=0  # choice the confusion pair index to display
)

#### task2 confusion pairs && misclassified and reference examples

In [ ]:
# Task 2 confusion pairs table
task2_confusion_pairs = get_confusion_pairs(
    task2_validation_predictions
)

display(task2_confusion_pairs.head(5))

In [ ]:
# show confused vs reference examples for Task 2
show_confused_vs_reference_examples(
    task2_validation_predictions,
    task2_confusion_pairs,
    image_folder=str(Task2_DIR),
    confusion_pair_index=0 # chice the confusion pair index to display
)

## 6. Feature Engineering

### Feature Experiment Preprocessing Utility

In [ ]:
def prepare_feature_experiment(
    metadata,
    feature_table,
    test_size
):

    # merge labels and features
    full_data = (
        metadata
        .merge(
            feature_table,
            on='image_id',
            how='inner'
        )
    )

    # separate features and labels
    X = full_data.drop(
        columns=[
            'image_id',
            'image_path',
            'class_id',
            'class_name'
        ]
    )

    y = full_data['class_name']

    # train validation split
    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    # standardisation
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_valid_scaled = scaler.transform(
        X_valid
    )

    return (
        X_train,
        X_valid,
        X_train_scaled,
        X_valid_scaled,
        y_train,
        y_valid,
        scaler
    )

### 6.1 Add HSV feature - summary statistics features

In [ ]:
# extract HSV features from one image
def extract_hsv_features(image_path):

    # read image
    image_bgr = cv2.imread(image_path)

    # convert BGR to HSV
    image_hsv = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2HSV
    )

    # extract mean and standard deviation
    hsv_features = {
        'h_mean': image_hsv[:, :, 0].mean(),
        'h_std': image_hsv[:, :, 0].std(),

        's_mean': image_hsv[:, :, 1].mean(),
        's_std': image_hsv[:, :, 1].std(),

        'v_mean': image_hsv[:, :, 2].mean(),
        'v_std': image_hsv[:, :, 2].std()
    }

    return hsv_features # simple mean and std features -> alternatively can use histogram features or texture features from skimage (e.g. graycomatrix, graycoprops) 

In [ ]:
# build HSV feature table for all images
def build_hsv_feature_table(
    metadata_table,
    task_folder
):

    # extract HSV features from all images
    feature_table = (

        (task_folder + '/' + metadata_table['image_path'])

        .apply(extract_hsv_features)

        .apply(pd.Series)

    )

    # combine image id and features
    return pd.concat(

        [
            metadata_table[['image_id']]
            .reset_index(drop=True),

            feature_table
        ],

        axis=1
    )

# build HSV features for task1
task1_hsv_train_features = build_hsv_feature_table(
    task1_train_metadata,
    'task1_data'
)

task1_hsv_test_features = build_hsv_feature_table(
    task1_test_metadata,
    'task1_data'
)


# build HSV features for task2
task2_hsv_train_features = build_hsv_feature_table(
    task2_train_metadata,
    'task2_data'
)

task2_hsv_test_features = build_hsv_feature_table(
    task2_test_metadata,
    'task2_data'
)


In [ ]:
task1_hsv_X_train, task1_hsv_X_valid, \
task1_hsv_X_train_scaled, task1_hsv_X_valid_scaled, \
task1_hsv_y_train, task1_hsv_y_valid, \
task1_hsv_scaler = prepare_feature_experiment(
    task1_train_metadata,
    task1_hsv_train_features,
    test_size=0.2
)

task2_hsv_X_train, task2_hsv_X_valid, \
task2_hsv_X_train_scaled, task2_hsv_X_valid_scaled, \
task2_hsv_y_train, task2_hsv_y_valid, \
task2_hsv_scaler = prepare_feature_experiment(
    task2_train_metadata,
    task2_hsv_train_features,
    test_size=0.25
)

In [ ]:
task1_hsv_model_results = run_baseline_models(
    task1_models,
    task1_hsv_X_train,
    task1_hsv_y_train,
    task1_hsv_X_valid,
    task1_hsv_y_valid,
    task1_hsv_X_train_scaled,
    task1_hsv_X_valid_scaled
)

task2_hsv_model_results = run_baseline_models(
    task2_models,
    task2_hsv_X_train,
    task2_hsv_y_train,
    task2_hsv_X_valid,
    task2_hsv_y_valid,
    task2_hsv_X_train_scaled,
    task2_hsv_X_valid_scaled
)

display_model_results(task1_hsv_model_results)
display_model_results(task2_hsv_model_results)

In [ ]:
feature_engineering_comparison = pd.DataFrame({

    'task': [
        'Task 1',
        'Task 1',
        'Task 2',
        'Task 2'
    ],

    'experiment': [
        'baseline_best',
        'hsv_feature_best',
        'baseline_best',
        'hsv_feature_best'
    ],

    'best_model': [

        task1_model_results.loc[
            task1_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task1_hsv_model_results.loc[
            task1_hsv_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task2_model_results.loc[
            task2_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task2_hsv_model_results.loc[
            task2_hsv_model_results['valid_f1'].idxmax(),
            'model_name'
        ]
    ],

    'valid_f1': [

        task1_model_results['valid_f1'].max(),

        task1_hsv_model_results['valid_f1'].max(),

        task2_model_results['valid_f1'].max(),

        task2_hsv_model_results['valid_f1'].max()
    ],

    'valid_accuracy': [

        task1_model_results['valid_accuracy'].max(),

        task1_hsv_model_results['valid_accuracy'].max(),

        task2_model_results['valid_accuracy'].max(),

        task2_hsv_model_results['valid_accuracy'].max()
    ]
})

display(feature_engineering_comparison)
print(feature_engineering_comparison.round(4).to_latex(index=False))

### 6.2 Selected feature - dimension reduction

In [ ]:
# select top k features

def select_k_features(
    X_train_scaled,
    X_valid_scaled,
    y_train,
    k
):

    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    X_selected_train = selector.fit_transform(
        X_train_scaled,
        y_train
    )

    X_selected_valid = selector.transform(
        X_valid_scaled)

    return X_selected_train, X_selected_valid, selector

In [ ]:
# test different k values

def evaluate_feature_selection_k(
    X_train_scaled,
    X_valid_scaled,
    y_train,
    y_valid,
    max_k
):

    k_results = []

    for k in range(10, max_k + 1, 10):

        X_selected_train, X_selected_valid, selector = select_k_features(
            X_train_scaled,
            X_valid_scaled,
            y_train,
            k
        )

        model = LogisticRegression(
            max_iter=2000,
            random_state=42
        )

        model.fit(
            X_selected_train,
            y_train
        )

        valid_pred = model.predict(
            X_selected_valid
        )

        valid_accuracy = accuracy_score(
            y_valid,
            valid_pred
        )

        k_results.append({
            'k': k,
            'valid_accuracy': valid_accuracy
        })

    return pd.DataFrame(k_results)

In [ ]:
# Task 1 feature selection curve

task1_k_results = evaluate_feature_selection_k(
    task1_X_train_scaled,
    task1_X_valid_scaled,
    task1_y_train,
    task1_y_valid,
    max_k=task1_X_train_scaled.shape[1]
)

display(task1_k_results)

In [ ]:
# Task 2 feature selection curve

task2_k_results = evaluate_feature_selection_k(
    task2_X_train_scaled,
    task2_X_valid_scaled,
    task2_y_train,
    task2_y_valid,
    max_k=task2_X_train_scaled.shape[1]
)

display(task2_k_results)

In [ ]:
# plot Task 1 feature selection performance

plt.figure(figsize=(8, 5))

plt.plot(
    task1_k_results['k'],
    task1_k_results['valid_accuracy'],
    marker='o',
    label='Feature Selection'
)

plt.axhline(
    y=task1_model_results['valid_accuracy'].max(),
    linestyle='--',
    label='Best Baseline'
)

plt.xlabel('Number of selected features (k)')
plt.ylabel('Validation accuracy')
plt.title('Task 1 Feature Selection Performance')
plt.legend()
plt.grid(True)
plt.show()
plt.savefig('T1 Feature performance.png')

In [ ]:
# plot Task 2 feature selection performance 

plt.figure(figsize=(8, 5))

plt.plot(
    task2_k_results['k'],
    task2_k_results['valid_accuracy'],
    marker='o',
    label='Feature Selection'
)

plt.axhline(
    y=task2_model_results['valid_accuracy'].max(),
    linestyle='--',
    label='Best Baseline'
)

plt.xlabel('Number of selected features (k)')
plt.ylabel('Validation accuracy')
plt.title('Task 2 Feature Selection Performance')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# get best k for each task

task1_best_k = int(
    task1_k_results.loc[
        task1_k_results['valid_accuracy'].idxmax(),
        'k'
    ]
)

task2_best_k = int(
    task2_k_results.loc[
        task2_k_results['valid_accuracy'].idxmax(),
        'k'
    ]
)

print('Task 1 best k:', task1_best_k)
print('Task 2 best k:', task2_best_k)

In [ ]:
# apply best k to Task 1

task1_X_selected_train, task1_X_selected_valid, task1_selector = select_k_features(
    task1_X_train_scaled,
    task1_X_valid_scaled,
    task1_y_train,
    task1_best_k
)

task1_selected_model_results = run_baseline_models(
    task1_models,
    task1_X_selected_train,
    task1_y_train,
    task1_X_selected_valid,
    task1_y_valid,
    task1_X_selected_train,
    task1_X_selected_valid
)

display_model_results(task1_selected_model_results)

In [ ]:
# apply best k to Task 2

task2_X_selected_train, task2_X_selected_valid, task2_selector = select_k_features(
    task2_X_train_scaled,
    task2_X_valid_scaled,
    task2_y_train,
    task2_best_k
)

task2_selected_model_results = run_baseline_models(
    task2_models,
    task2_X_selected_train,
    task2_y_train,
    task2_X_selected_valid,
    task2_y_valid,
    task2_X_selected_train,
    task2_X_selected_valid
)

display_model_results(task2_selected_model_results)

In [ ]:
# compare baseline vs feature selection

feature_selection_comparison = pd.DataFrame({
    'task': [
        'Task 1',
        'Task 1',
        'Task 2',
        'Task 2'
    ],
    'experiment': [
        'baseline_best',
        'feature_selection_best',
        'baseline_best',
        'feature_selection_best'
    ],
    'best_k': [
        '-',
        task1_best_k,
        '-',
        task2_best_k
    ],
    'valid_f1': [
        task1_model_results['valid_f1'].max(),
        task1_selected_model_results['valid_f1'].max(),
        task2_model_results['valid_f1'].max(),
        task2_selected_model_results['valid_f1'].max()
    ],
    'valid_accuracy': [
        task1_model_results['valid_accuracy'].max(),
        task1_selected_model_results['valid_accuracy'].max(),
        task2_model_results['valid_accuracy'].max(),
        task2_selected_model_results['valid_accuracy'].max()
    ]
})

display(feature_selection_comparison)
print(feature_selection_comparison.round(4).to_latex(index=False))

In [ ]:
# run baseline models with different k values and store results
task1_feature_selection_results = []

for k in range(10, 221, 10):

    # select features
    selector = SelectKBest(
        score_func=f_classif,
        k=k
    )

    task1_X_selected_train = selector.fit_transform(
        task1_X_train_scaled,
        task1_y_train
    )

    task1_X_selected_valid = selector.transform(
        task1_X_valid_scaled
    )


    # run baseline models
    selected_results = run_baseline_models(
        task1_models,

        task1_X_selected_train,
        task1_y_train,

        task1_X_selected_valid,
        task1_y_valid,

        task1_X_selected_train,
        task1_X_selected_valid
    )


    # store results
    selected_results['k'] = k

    task1_feature_selection_results.append(
        selected_results
    )


# combine all results
task1_feature_selection_results = pd.concat(
    task1_feature_selection_results,
    axis=0
).reset_index(drop=True)

In [ ]:
task1_feature_selection_results.groupby(
    'k'
)['valid_f1'].mean()

### 6.3 Local Feature Extraction

In [ ]:
# extract foreground-focused features
def extract_background_suppressed_features(image_path):

    # read image
    image_bgr = cv2.imread(image_path)

    # convert BGR to HSV
    image_hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)

    # build saturation mask to find foreground region
    saturation_mask = (image_hsv[:, :, 1] > 40).astype(np.uint8) * 255

    # get foreground points
    foreground_points = cv2.findNonZero(saturation_mask)

    # if no foreground found, use original image
    if foreground_points is None:

        foreground_bgr = image_bgr

    else:

        # get bounding box around foreground
        x, y, w, h = cv2.boundingRect(foreground_points)

        # crop foreground region
        foreground_bgr = image_bgr[y:y + h, x:x + w]

        # avoid invalid tiny crops
        if foreground_bgr.shape[0] < 2 or foreground_bgr.shape[1] < 2:

            foreground_bgr = image_bgr

    # convert cropped foreground to HSV and grayscale
    foreground_hsv = cv2.cvtColor(foreground_bgr, cv2.COLOR_BGR2HSV)

    foreground_gray = cv2.cvtColor(
        foreground_bgr,
        cv2.COLOR_BGR2GRAY,
    )

    # reduce grayscale levels
    foreground_gray = (foreground_gray // 8).astype(np.uint8)

    # build GLCM texture matrix
    foreground_glcm = graycomatrix(
        foreground_gray,
        distances=[1],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=32,
        symmetric=True,
        normed=True,
    )

    # return foreground features
    return {

        # HSV summary statistics
        'fg_h_mean': foreground_hsv[:, :, 0].mean(),
        'fg_h_std': foreground_hsv[:, :, 0].std(),

        'fg_s_mean': foreground_hsv[:, :, 1].mean(),
        'fg_s_std': foreground_hsv[:, :, 1].std(),

        'fg_v_mean': foreground_hsv[:, :, 2].mean(),
        'fg_v_std': foreground_hsv[:, :, 2].std(),

        # texture features
        'fg_texture_contrast':
            graycoprops(foreground_glcm, 'contrast').mean(),

        'fg_texture_homogeneity':
            graycoprops(foreground_glcm, 'homogeneity').mean(),

        'fg_texture_energy':
            graycoprops(foreground_glcm, 'energy').mean(),

        'fg_texture_correlation':
            graycoprops(foreground_glcm, 'correlation').mean(),
    }


In [ ]:
# build feature table
def build_background_suppressed_feature_table(
    metadata_table,
    task_folder,
):

    # extract features from all images
    feature_table = (
        (task_folder + '/' + metadata_table['image_path'])
        .apply(extract_background_suppressed_features)
        .apply(pd.Series)
    )

    # combine image_id with extracted features
    return pd.concat(
        [
            metadata_table[['image_id']].reset_index(drop=True),
            feature_table,
        ],
        axis=1,
    )

In [ ]:
# build background-suppressed feature tables

task1_bg_train_features = build_background_suppressed_feature_table(
    task1_train_metadata,
    'task1_data'
)

task1_bg_test_features = build_background_suppressed_feature_table(
    task1_test_metadata,
    'task1_data'
)

task2_bg_train_features = build_background_suppressed_feature_table(
    task2_train_metadata,
    'task2_data'
)

task2_bg_test_features = build_background_suppressed_feature_table(
    task2_test_metadata,
    'task2_data'
)

In [ ]:
# merge HSV and background-suppressed features

task1_train_engineered = (
    task1_train_full
    .merge(task1_hsv_train_features, on='image_id')
    .merge(task1_bg_train_features, on='image_id')
)

task1_test_engineered = (
    task1_test_full
    .merge(task1_hsv_test_features, on='image_id')
    .merge(task1_bg_test_features, on='image_id')
)

task2_train_engineered = (
    task2_train_full
    .merge(task2_hsv_train_features, on='image_id')
    .merge(task2_bg_train_features, on='image_id')
)

task2_test_engineered = (
    task2_test_full
    .merge(task2_hsv_test_features, on='image_id')
    .merge(task2_bg_test_features, on='image_id')
)

In [ ]:
# merge background-suppressed features

task1_train_bg_engineered = (
    task1_train_full
    .merge(task1_bg_train_features, on='image_id')
)

task1_test_bg_engineered = (
    task1_test_full
    .merge(task1_bg_test_features, on='image_id')
)

task2_train_bg_engineered = (
    task2_train_full
    .merge(task2_bg_train_features, on='image_id')
)

task2_test_bg_engineered = (
    task2_test_full
    .merge(task2_bg_test_features, on='image_id')
)

# create feature engineering summary

feature_engineering_summary = pd.DataFrame([
    {
        'table': 'task1_train_engineered',
        'technique': 'global_hsv_texture',
        'rows': task1_train_engineered.shape[0],
        'columns': task1_train_engineered.shape[1]
    },
    {
        'table': 'task1_test_engineered',
        'technique': 'global_hsv_texture',
        'rows': task1_test_engineered.shape[0],
        'columns': task1_test_engineered.shape[1]
    },
    {
        'table': 'task2_train_engineered',
        'technique': 'global_hsv_texture',
        'rows': task2_train_engineered.shape[0],
        'columns': task2_train_engineered.shape[1]
    },
    {
        'table': 'task2_test_engineered',
        'technique': 'global_hsv_texture',
        'rows': task2_test_engineered.shape[0],
        'columns': task2_test_engineered.shape[1]
    },
    {
        'table': 'task1_train_bg_engineered',
        'technique': 'background_suppressed_hsv_texture',
        'rows': task1_train_bg_engineered.shape[0],
        'columns': task1_train_bg_engineered.shape[1]
    },
    {
        'table': 'task1_test_bg_engineered',
        'technique': 'background_suppressed_hsv_texture',
        'rows': task1_test_bg_engineered.shape[0],
        'columns': task1_test_bg_engineered.shape[1]
    },
    {
        'table': 'task2_train_bg_engineered',
        'technique': 'background_suppressed_hsv_texture',
        'rows': task2_train_bg_engineered.shape[0],
        'columns': task2_train_bg_engineered.shape[1]
    },
    {
        'table': 'task2_test_bg_engineered',
        'technique': 'background_suppressed_hsv_texture',
        'rows': task2_test_bg_engineered.shape[0],
        'columns': task2_test_bg_engineered.shape[1]
    }
])

# display(feature_engineering_summary)

In [ ]:
# Task 1: split background-suppressed engineered features

task1_bg_X = task1_train_bg_engineered.drop(
    columns=['image_id', 'image_path', 'class_id', 'class_name']
)

task1_bg_y = task1_train_bg_engineered['class_name']

task1_bg_X_train, task1_bg_X_valid, task1_bg_y_train, task1_bg_y_valid = train_test_split(
    task1_bg_X,
    task1_bg_y,
    test_size=0.2,
    random_state=42,
    stratify=task1_bg_y
)


# Task 2

task2_bg_X = task2_train_bg_engineered.drop(
    columns=['image_id', 'image_path', 'class_id', 'class_name']
)

task2_bg_y = task2_train_bg_engineered['class_name']

task2_bg_X_train, task2_bg_X_valid, task2_bg_y_train, task2_bg_y_valid = train_test_split(
    task2_bg_X,
    task2_bg_y,
    test_size=0.25,
    random_state=42,
    stratify=task2_bg_y
)

In [ ]:
# standardise

task1_bg_scaler = StandardScaler()

task1_bg_X_train_scaled = task1_bg_scaler.fit_transform(task1_bg_X_train)
task1_bg_X_valid_scaled = task1_bg_scaler.transform(task1_bg_X_valid)


task2_bg_scaler = StandardScaler()

task2_bg_X_train_scaled = task2_bg_scaler.fit_transform(task2_bg_X_train)
task2_bg_X_valid_scaled = task2_bg_scaler.transform(task2_bg_X_valid)

In [ ]:
# evaluate background-suppressed features

task1_bg_model_results = run_baseline_models(
    task1_models,
    task1_bg_X_train,
    task1_bg_y_train,
    task1_bg_X_valid,
    task1_bg_y_valid,
    task1_bg_X_train_scaled,
    task1_bg_X_valid_scaled
)

task2_bg_model_results = run_baseline_models(
    task2_models,
    task2_bg_X_train,
    task2_bg_y_train,
    task2_bg_X_valid,
    task2_bg_y_valid,
    task2_bg_X_train_scaled,
    task2_bg_X_valid_scaled
)

display_model_results(task1_bg_model_results)
display_model_results(task2_bg_model_results)

In [ ]:
feature_engineering_comparison = pd.DataFrame({
    'task': ['Task 1', 'Task 1', 'Task 2', 'Task 2'],
    'experiment': [
        'baseline_best',
        'background_suppressed_best',
        'baseline_best',
        'background_suppressed_best'
    ],
    'valid_f1': [
        task1_model_results['valid_f1'].max(),
        task1_bg_model_results['valid_f1'].max(),
        task2_model_results['valid_f1'].max(),
        task2_bg_model_results['valid_f1'].max()
    ]
})

display(feature_engineering_comparison)

### 6.4 comparison table

In [ ]:
all_experiment_results = pd.DataFrame({

    'task': [

        'Task 1',
        'Task 1',
        'Task 1',
        'Task 1',

        'Task 2',
        'Task 2',
        'Task 2',
        'Task 2'
    ],

    'experiment_order': [

        1,
        2,
        3,
        4,

        1,
        2,
        3,
        4
    ],

    'experiment': [

        'baseline',
        'hsv_feature',
        'feature_selection',
        'feature_extraction',

        'baseline',
        'hsv_feature',
        'feature_selection',
        'feature_extraction'
    ],

    'best_model': [

        # Task 1 baseline
        task1_model_results.loc[
            task1_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 1 HSV
        task1_hsv_model_results.loc[
            task1_hsv_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 1 feature selection
        task1_selected_model_results.loc[
            task1_selected_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 1 feature extraction
        task1_bg_model_results.loc[
            task1_bg_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 2 baseline
        task2_model_results.loc[
            task2_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 2 HSV
        task2_hsv_model_results.loc[
            task2_hsv_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 2 feature selection
        task2_selected_model_results.loc[
            task2_selected_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        # Task 2 feature extraction
        task2_bg_model_results.loc[
            task2_bg_model_results['valid_f1'].idxmax(),
            'model_name'
        ]
    ],

    'valid_f1': [

        # Task 1 baseline
        task1_model_results['valid_f1'].max(),

        # Task 1 HSV
        task1_hsv_model_results['valid_f1'].max(),

        # Task 1 feature selection
        task1_selected_model_results['valid_f1'].max(),

        # Task 1 feature extraction
        task1_bg_model_results['valid_f1'].max(),

        # Task 2 baseline
        task2_model_results['valid_f1'].max(),

        # Task 2 HSV
        task2_hsv_model_results['valid_f1'].max(),

        # Task 2 feature selection
        task2_selected_model_results['valid_f1'].max(),

        # Task 2 feature extraction
        task2_bg_model_results['valid_f1'].max()
    ],

    'valid_accuracy': [

        # Task 1 baseline
        task1_model_results['valid_accuracy'].max(),

        # Task 1 HSV
        task1_hsv_model_results['valid_accuracy'].max(),

        # Task 1 feature selection
        task1_selected_model_results['valid_accuracy'].max(),

        # Task 1 feature extraction
        task1_bg_model_results['valid_accuracy'].max(),

        # Task 2 baseline
        task2_model_results['valid_accuracy'].max(),

        # Task 2 HSV
        task2_hsv_model_results['valid_accuracy'].max(),

        # Task 2 feature selection
        task2_selected_model_results['valid_accuracy'].max(),

        # Task 2 feature extraction
        task2_bg_model_results['valid_accuracy'].max()
    ]
})


# sort table
all_experiment_results = (
    all_experiment_results
    .sort_values(
        ['task', 'experiment_order']
    )
    .reset_index(drop=True)
)


# display results
display(all_experiment_results)

### 6.5 combine selection + extraction

In [ ]:
# choose best k from previous feature selection
task1_bg_best_k = task1_best_k
task2_bg_best_k = task2_best_k


# Task 1: select features from background-suppressed engineered features
task1_bg_selected_train, task1_bg_selected_valid, task1_bg_selector = select_k_features(
    task1_bg_X_train_scaled,
    task1_bg_X_valid_scaled,
    task1_bg_y_train,
    task1_bg_best_k
)


# Task 2
task2_bg_selected_train, task2_bg_selected_valid, task2_bg_selector = select_k_features(
    task2_bg_X_train_scaled,
    task2_bg_X_valid_scaled,
    task2_bg_y_train,
    task2_bg_best_k
)

In [ ]:
# evaluate Task 1 extraction + selection

task1_bg_selected_model_results = run_baseline_models(
    task1_models,
    task1_bg_selected_train,
    task1_bg_y_train,
    task1_bg_selected_valid,
    task1_bg_y_valid,
    task1_bg_selected_train,
    task1_bg_selected_valid
)

display_model_results(task1_bg_selected_model_results)

In [ ]:
# evaluate Task 2 extraction + selection

task2_bg_selected_model_results = run_baseline_models(
    task2_models,
    task2_bg_selected_train,
    task2_bg_y_train,
    task2_bg_selected_valid,
    task2_bg_y_valid,
    task2_bg_selected_train,
    task2_bg_selected_valid
)

display_model_results(task2_bg_selected_model_results)

In [ ]:
combined_feature_results = pd.DataFrame({

    'task': [
        'Task 1',
        'Task 1',
        'Task 2',
        'Task 2'
    ],

    'experiment': [
        'feature_extraction',
        'extraction_plus_selection',
        'feature_extraction',
        'extraction_plus_selection'
    ],

    'best_model': [
        task1_bg_model_results.loc[
            task1_bg_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task1_bg_selected_model_results.loc[
            task1_bg_selected_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task2_bg_model_results.loc[
            task2_bg_model_results['valid_f1'].idxmax(),
            'model_name'
        ],

        task2_bg_selected_model_results.loc[
            task2_bg_selected_model_results['valid_f1'].idxmax(),
            'model_name'
        ]
    ],

    'valid_f1': [
        task1_bg_model_results['valid_f1'].max(),
        task1_bg_selected_model_results['valid_f1'].max(),
        task2_bg_model_results['valid_f1'].max(),
        task2_bg_selected_model_results['valid_f1'].max()
    ],

    'valid_accuracy': [
        task1_bg_model_results['valid_accuracy'].max(),
        task1_bg_selected_model_results['valid_accuracy'].max(),
        task2_bg_model_results['valid_accuracy'].max(),
        task2_bg_selected_model_results['valid_accuracy'].max()
    ]
})

display(combined_feature_results)

## 7. hyperparametr tuning 

### 7.1 Random Forest GridSearchCV

In [ ]:
# parameter search space
task2_tuning_grid = {

    # number of trees
    'n_estimators': [100, 200],

    # maximum tree depth
    'max_depth': [None, 10, 20],

    # minimum samples in leaf node
    'min_samples_leaf': [1, 2, 4],

    # number of features used at each split
    'max_features': ['sqrt', 0.5],
}

# stratified cross validation
task2_tuning_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

# grid search
task2_grid_search = GridSearchCV(

    estimator=RandomForestClassifier(
        random_state=42,
    ),

    param_grid=task2_tuning_grid,

    scoring='f1_macro',

    cv=task2_tuning_cv,

    n_jobs=-1,
)

# fit grid search
task2_grid_search.fit(
    task2_X_train_scaled,
    task2_y_train,
)


### 7.2 Best Parameters

In [ ]:
# display best parameters and best cross-validation score

print(
    'Best parameters:',
    task2_grid_search.best_params_
)

print(
    'Best cross-validation macro F1:',
    round(task2_grid_search.best_score_, 4)
)

In [ ]:
# convert grid search results to table

task2_grid_results = pd.DataFrame(
    task2_grid_search.cv_results_
)

task2_grid_results_display = (
    task2_grid_results[
        [
            'mean_test_score',
            'std_test_score',
            'param_n_estimators',
            'param_max_depth',
            'param_min_samples_leaf',
            'param_max_features'
        ]
    ]
    .sort_values(
        'mean_test_score',
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    task2_grid_results_display.head(10)
)

### 7.3 Tuned vs Baseline Comparison

In [ ]:
# evaluate tuned Random Forest on validation set

task2_tuned_rf_result = fit_and_evaluate_model(
    task2_grid_search.best_estimator_,
    task2_X_train,
    task2_y_train,
    task2_X_valid,
    task2_y_valid,
    'tuned_random_forest'
)

display(
    task2_tuned_rf_result.drop(
        columns=[
            'fitted_model',
            'class_results'
        ]
    )
)

## 8. ensemble - final model

### 8.1 EfficientNet-B4 Feature Extraction

In [ ]:
# Load EfficientNet-B4 (Image pretrained, classification removed)
backbone_t1 = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
backbone_t1.classifier = torch.nn.Identity()
backbone_t1.eval()

norm_t1 = T.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])

task1_tta_transforms = [
    T.Compose([T.Resize((380, 380)), T.CenterCrop(320),  T.ToTensor(),norm_t1]),
    T.Compose([T.Resize((400, 400)), T.CenterCrop(320), T.ToTensor(),norm_t1]),
    T.Compose([T.Resize((380, 380)), T.CenterCrop(320), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm_t1]),
    T.Compose([T.Resize((400, 400)), T.CenterCrop(320), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm_t1]),
]

def extract_efficientnet_features(image_paths, data_dir, use_tta=True):
    tfs = task1_tta_transforms if use_tta else [task1_tta_transforms[0]]
    features = []
    for i, rel_path in enumerate(image_paths):
        image_path = data_dir / rel_path
        try:
            image = Image.open(image_path).convert("RGB")
        except (OSError, ValueError) as exc:
            raise RuntimeError(f"Cannot read image: {image_path}") from exc
        aug_feats = []
        for tf in tfs:
            tensor = tf(image).unsqueeze(0)
            with torch.no_grad():
                feature = backbone_t1(tensor).squeeze().numpy()
            aug_feats.append(feature)
        features.append(np.mean(aug_feats, axis=0))
        if (i + 1) % 200 == 0:
            print(f"{i + 1}/{len(image_paths)} processed")
    return np.array(features)

In [ ]:
# Extract features for Task 1 train and test sets
print("Extracting Task 1 test features(TTA *4)...")
task1_effnet_X_train_raw = extract_efficientnet_features(
    task1_train_metadata["image_path"].tolist(), Task1_DIR, use_tta=True
)

print("\nExtracting Task 1 test features(TTA *4)...")
task1_effnet_X_test_raw = extract_efficientnet_features(
    task1_test_metadata["image_path"].tolist(), Task1_DIR, use_tta=True
)

In [ ]:
# Standardise features
task1_scaler = StandardScaler()
task1_effnet_X_train = (task1_scaler.fit_transform(task1_effnet_X_train_raw))
task1_effnet_X_test = (task1_scaler.transform(task1_effnet_X_test_raw))
task1_effnet_y_train = task1_train_metadata["class_id"].values

print(f"Feature shape: train={task1_effnet_X_train.shape}, test ={task1_effnet_X_test.shape}")

### 8.2 Task1 - Model Training & Cross_Validation

In [ ]:
# Define models: SVM (C=20, C=50) + Logistic Regression
task1_effnet_models = {
    "svm_c20": SVC(kernel="rbf", C=20, gamma="scale", probability=True, random_state=42),
    "svm_c50": SVC(kernel="rbf", C=50, gamma="scale", probability=True, random_state=42),
    "logistic_regression": LogisticRegression(C=10.0, max_iter=2000, solver="lbfgs", random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
task1_effnet_cv_scores = {}

print("\nCross-validation results (5-fold):")
for model_name, model in task1_effnet_models.items():
    scores = cross_val_score(
        model, task1_effnet_X_train, task1_effnet_y_train,
        cv=cv, scoring="accuracy"
    )
    task1_effnet_cv_scores[model_name] = scores
    print(f"{model_name}: {scores.mean():.4f} +/- {scores.std():.4f}")


### 8.3 Task1 - Weighted Soft Voting Ensemble & Submission

In [ ]:
# Three distinct models: SVM-RBF (C=20), SVM-RBF(C=50), Logistic Regression
print("Training models on full training set...")
for model_name, model in task1_effnet_models.items():
    model.fit(task1_effnet_X_train, task1_effnet_y_train)
    print(f"{model_name} done")

In [ ]:
# Weighted soft voting: weights proportional to CV accuracy
weight_sum = sum(task1_effnet_cv_scores[n].mean() for n in task1_effnet_models)
task1_effnet_weights = {
    n: task1_effnet_cv_scores[n].mean() / weight_sum
    for n in task1_effnet_models
}
print("Ensemble weights:", task1_effnet_weights)

task1_effnet_ensemble_proba = sum(
    task1_effnet_weights[name] * model.predict_proba(task1_effnet_X_test)
    for name, model in task1_effnet_models.items()
)
task1_effnet_pred = np.argmax(task1_effnet_ensemble_proba,axis=1)

In [ ]:
# Create Kaggle submission
task1_effnet_submission = pd.DataFrame({
    "image_id": task1_test_metadata["image_id"],
    "class_id": task1_effnet_pred
})
task1_effnet_submission.to_csv("task1_effnet_submission.csv", index=False)
print("Submission saved: task1_submission.csv")
display(task1_effnet_submission.head())

### 8.4 Task 2 - EfficientNet-B2 Feature Extraction (TTA *7)

In [ ]:
# Load EfficientNet-B2 (lighter than B4, suits small Task2 dataset)
backbone_t2 = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
backbone_t2.classifier = torch.nn.Identity()
backbone_t2.eval()

norm_t2 = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

task2_tta_transforms = [
    T.Compose([T.Resize((260, 260)), T.CenterCrop(224), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((260, 260)), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((288, 288)), T.CenterCrop(224), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((288, 288)), T.CenterCrop(224), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((260, 260)), T.CenterCrop(224), T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((260, 260)), T.CenterCrop(224), T.RandomRotation(degrees=15), T.ToTensor(), norm_t2]),
    T.Compose([T.Resize((288, 288)), T.CenterCrop(224), T.ColorJitter(brightness=0.2, saturation=0.2), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), norm_t2]),
]


def extract_b2_features(image_paths, data_dir, use_tta=True):
    tfs = task2_tta_transforms if use_tta else [task2_tta_transforms[0]]
    feats = []
    for i, rel_path in enumerate(image_paths):
        path = data_dir / rel_path
        try:
            img = Image.open(path).convert("RGB")
        except (OSError, ValueError) as exc:
            raise RuntimeError(f"Cannot read image: {path}") from exc
        aug_feats = []
        for tf in tfs:
            tensor = tf(img).unsqueeze(0)
            with torch.no_grad():
                f = backbone_t2(tensor).squeeze().numpy()
            aug_feats.append(f)
        feats.append(np.mean(aug_feats, axis=0))
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(image_paths)}")
    return np.array(feats)

In [ ]:
#Extract features for Task 2
print("\nExtracting train features (TTA x7)...")
task2_effnet_X_train_raw = extract_b2_features(
    task2_train_metadata["image_path"].tolist(), Task2_DIR, use_tta = True
)

print("\nExtracting test features (TTA x7)...")
task2_effnet_X_test_raw = extract_b2_features(
    task2_test_metadata["image_path"].tolist(), Task2_DIR, use_tta = True
)

In [ ]:
# Standardise + PCA dimensionality reduction
# PCA n=200 chosen based on cross-validation sweep across [80, 100, 120, 150, 200]
task2_scaler = StandardScaler()
task2_effnet_X_scaled = task2_scaler.fit_transform(task2_effnet_X_train_raw)
task2_effnet_X_test_scaled = task2_scaler.transform(task2_effnet_X_test_raw)

task2_effnet_pca = PCA(n_components=200, random_state=42)
task2_effnet_X_train = task2_effnet_pca.fit_transform(task2_effnet_X_scaled)
task2_effnet_X_test = task2_effnet_pca.transform(task2_effnet_X_test_scaled)
task2_effnet_y_train = task2_train_metadata["class_id"].values
print(f"After PCA: {task2_effnet_X_train.shape[1]} dims, variance explained: {task2_effnet_pca.explained_variance_ratio_.sum():.3f}")

## 8.5 Task 2 - Cross-Validation & Model Selection

In [ ]:
# Four distinct models for Task 2 : SVM Linear, SVM RBF, LR, ensemble
task2_effnet_models = {
    "SVM_L_C1": SVC(kernel="linear", C=1.0, probability=True, random_state=42),
    "SVM_L_C5": SVC(kernel="linear", C=5.0, probability=True, random_state=42),
    "SVM_RBF": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42),
    "LR_C1": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", random_state=42),
    "LR_C5": LogisticRegression(C=5.0, max_iter=1000, solver="lbfgs", random_state=42),
}

task2_effnet_cv_scores = {}

print("\n── Cross-validation results ──")

for name, model in task2_effnet_models.items():
    scores = cross_val_score(model, task2_effnet_X_train, task2_effnet_y_train, cv=cv, scoring="accuracy")
    task2_effnet_cv_scores[name] = scores
    print(f"  {name:12s}: {scores.mean():.4f} ± {scores.std():.4f}")

## 8.6 Task2 - Weighted Soft Voting Ensemble

In [ ]:
# Train all models on full training set
print("\nTraining all models on full training set...")
for name, model in task2_effnet_models.items():
    model.fit(task2_effnet_X_train, task2_effnet_y_train)

In [ ]:
# Weighted soft voting ensemble (weight proportional to CV accuracy)
task2_weight_sum = sum(task2_effnet_cv_scores[n].mean() for n in task2_effnet_models)
task2_effnet_weights = { n: task2_effnet_cv_scores[n].mean() / task2_weight_sum
                         for n in task2_effnet_models
                         }
print("Ensemble weights:")
for name, w in task2_effnet_weights.items():
    print(f"  {name:12s}: {w:.3f}")

task2_ensemble_proba = sum(
    task2_effnet_weights[name] * model.predict_proba(task2_effnet_X_test)
    for name, model in task2_effnet_models.items()
)
task2_pred = np.argmax(task2_ensemble_proba, axis=1)

## 8.7 Task 2 Error Analysis

In [ ]:
ea2_train_idx, ea2_val_idx = train_test_split(
    np.arange(len(task2_effnet_X_train)),test_size = 0.2, stratify = task2_effnet_y_train, random_state=42
)
ea2_models = {name: clone(model) for name, model in task2_effnet_models.items()}
for name,model in ea2_models.items():
    model.fit(task2_effnet_X_train[ea2_train_idx], task2_effnet_y_train[ea2_train_idx])

ea2_ensemble_proba = sum(
    task2_effnet_weights[name] * model.predict_proba(task2_effnet_X_train[ea2_val_idx])
    for name, model in ea2_models.items()
)
ea2_pred = np.argmax(ea2_ensemble_proba, axis=1)

task2_id_to_class = task2_train_metadata.set_index('class_id')['class_name'].to_dict()
task2_effnet_val_preds = pd.DataFrame({
    'image_path':   task2_train_metadata['image_path'].iloc[ea2_val_idx].values,
    'true_class':   [task2_id_to_class[y] for y in task2_effnet_y_train[ea2_val_idx]],
    'predicted_class':  [task2_id_to_class[y] for y in ea2_pred],
})
task2_effnet_val_preds['is_correct'] = (
    task2_effnet_val_preds['true_class'] == task2_effnet_val_preds['predicted_class']
)
print(f"Ensemble holdout accuracy: {task2_effnet_val_preds['is_correct'].mean():.4f}")

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    task2_effnet_val_preds['true_class'],
    task2_effnet_val_preds['predicted_class'],
    title='Task 2 Confusion Matrix — EfficientNet-B2 + Weighted Soft Voting Ensemble (holdout)',
    cmap='Oranges',
    figsize=(13, 11),
    save_path='task2_effnet_holdout_confusion_matrix.png'
)

In [ ]:
# Task 2 confusion pairs — highlight known hard pairs
task2_effnet_confusion_pairs = get_confusion_pairs(task2_effnet_val_preds)
display(task2_effnet_confusion_pairs.head(10))

# Highlight the hardest species pairs (expected from visual similarity)
hard_pairs = ['House_Sparrow', 'Song_Sparrow', 'Herring_Gull', 'Ring_billed_Gull',
              'Yellow_Warbler', 'Wilson_Warbler']
hard_confusions = task2_effnet_confusion_pairs[
    task2_effnet_confusion_pairs['true_class'].isin(hard_pairs) |
    task2_effnet_confusion_pairs['predicted_class'].isin(hard_pairs)
]
print("\nKey visually-similar confusions:")
display(hard_confusions)

## 8.8 Kaggle Submission

In [ ]:
# Generate Kaggle submission
task2_submission = pd.DataFrame({
    "image_id": task2_test_metadata["image_id"],
    "class_id": task2_pred,
})
task2_submission.to_csv("task2_submission.csv", index=False)
print(f"\nDone! Saved task2_submission.csv ({len(task2_submission)} rows)")
display(task2_submission.head(10))

## 9. Summary - Model Comparison Across All Experiment

In [ ]:
summary_data = {
    'Experiment': [
        # Task 1
        'Task 1 - Logistic Regression',
        'Task 1 — Linear SVM',
        'Task 1 — RBF SVM',
        'Task 1 — Random Forest',
        # Task 2
        'Task 2 — Logistic Regression',
        'Task 2 — Linear SVM',
        'Task 2 — RBF SVM',
        'Task 2 — EfficientNet-B2 + Ensemble',
    ],
    'Validation accuracy': [
        # Fill in from your actual results
        task1_model_results.loc[task1_model_results['model_name']=='logistic_regression','valid_accuracy'].values[0],
        task1_model_results.loc[task1_model_results['model_name']=='linear_svm','valid_accuracy'].values[0],
        task1_model_results.loc[task1_model_results['model_name']=='rbf_svm','valid_accuracy'].values[0],
        task1_model_results.loc[task1_model_results['model_name']=='random_forest','valid_accuracy'].values[0],

        task2_model_results.loc[task2_model_results['model_name']=='logistic_regression','valid_accuracy'].values[0],
        task2_model_results.loc[task2_model_results['model_name']=='linear_svm','valid_accuracy'].values[0],
        task2_model_results.loc[task2_model_results['model_name']=='rbf_svm','valid_accuracy'].values[0],
        float(task2_effnet_val_preds['is_correct'].mean()),
    ],
    'Task': ['Task 1']*4 + ['Task 2']*4,
    'Evaluation': ['Original baseline validation split']*7 + ['Error-analysis holdout (selection-dependent)'],
}

summary_df = pd.DataFrame(summary_data)
summary_df['Validation accuracy'] = summary_df['Validation accuracy'].round(4)
display(summary_df)
print(summary_df.round(4).to_latex(index=False))

In [ ]:
# Bar chart — Task 1 vs Task 2 accuracy across experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, task in zip(axes, ['Task 1', 'Task 2']):
    subset = summary_df[summary_df['Task'] == task]
    labels = [e.split(' — ')[-1].split(' - ')[-1] for e in subset['Experiment']]
    bars = ax.barh(labels, subset['CV / Val Accuracy'], color='steelblue')
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Accuracy')
    ax.set_title(f'{task} — Model Comparison')
    for bar, val in zip(bars, subset['CV / Val Accuracy']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Kaggle threshold')
    ax.legend(fontsize=8)

plt.suptitle('Model Accuracy Comparison — Baseline to Deep Features', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")